# AI Co-Scientist: Evolving a Formula

* * * 

<div class="alert alert-success">  
    
### Learning Objectives 
    
* Explain how an evaluator turns a hypothesis into a score.
* Distinguish training error from held-out error.
* Write a candidate program that the evaluator can score.
* Run an evolutionary search for a formula without an API key.
</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
🥊 **Challenge**: Interactive exercise. We'll work through these in the workshop!<br>
⏱️ **Take-home**: A challenge we skip in the 1-hour workshop. Try it afterwards; answers are in `solutions/`.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
🎬 **Demo**: Showing off something more advanced – so you know what Python can be used for!<br>

### Sections
1. [The Evaluator Is the Science](#evaluator)
2. [Evolution Without a Language Model](#mini)
3. [Demo: OpenEvolve](#demo)

In [ ]:
# Imports and paths used in this notebook.
import os
import sys
from pathlib import Path

ROOT = Path('..').resolve()
SCRIPTS = ROOT / 'lessons' / 'scripts' / '02_openevolve'
sys.path.insert(0, str(SCRIPTS))
from evaluator import evaluate

<a id='evaluator'></a>

# The Evaluator Is the Science

[AlphaEvolve](https://deepmind.google/discover/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/)
and its open-source cousin [OpenEvolve](https://github.com/algorithmicsuperintelligence/openevolve)
work like this:

1. A language model rewrites a program.
2. An **evaluator** runs the program and gives it a score.
3. The best programs are kept and rewritten again.

The language model can be creative, but the evaluator decides what counts as
better. So writing the evaluator is the scientist's job.

Here a candidate program defines `fit(train_rows)`. It gets the training rows
and returns a function `predict(x1, x2)`. Here is the starting program, a
deliberately naive law:

In [ ]:
print((SCRIPTS / 'initial_program.py').read_text())

The evaluator in `scripts/02_openevolve/evaluator.py` fits the candidate on the
training groups, then measures its error on both splits. The score that drives
evolution is `combined_score = 1 / (1 + test error)`: 1.0 is perfect, and an
error of a factor of 10 scores 0.5.

In [ ]:
evaluate(str(SCRIPTS / 'initial_program.py'))

🔔 **Question:** Why does the score use the held-out groups, and not the
training groups?

## 🥊 Challenge 1: A Leaky Evaluator

Here is a candidate that cheats: it memorizes the training data. For any
`x1`, it returns the `y` of the training row whose `x1` is closest.

Fill in `predict()`, then run the next cell to score it. Compare its training
error with its test error. What would happen if evolution used the training
error as its score?

💡 **Tip:** `min(items, key=some_function)` returns the item for which
`some_function` is smallest.

In [ ]:
%%writefile memorizer.py
import math


def fit(train_rows):
    def predict(x1, x2):
        nearest = ...
        return ...

    return predict

In [ ]:
evaluate('memorizer.py')

## 🥊 Challenge 2: Write a Better Candidate

⏱️ **Take-home:** we skip this challenge in the 1-hour workshop. Try it afterwards.

Write a candidate that fits `y = c * x1^p * x2^q`, with all three constants
learned from the training rows.

In log space this is a straight-line fit with two inputs:
`log10(y) = log10(c) + p * log10(x1) + q * log10(x2)`. `np.linalg.lstsq()`
solves it. Can you beat a score of 0.95?

In [ ]:
%%writefile my_candidate.py
import math

import numpy as np


def fit(train_rows):
    design = np.array([[1, math.log10(row['x1']), math.log10(row['x2'])] for row in train_rows])
    target = np.array([math.log10(row['y']) for row in train_rows])
    # YOUR CODE HERE
    ...

    def predict(x1, x2):
        return ...

    return predict

In [ ]:
evaluate('my_candidate.py')

<a id='mini'></a>

# Evolution Without a Language Model

`scripts/02_openevolve/mini_evolve.py` keeps the core idea of evolution and
drops the language model, so it runs in seconds with no key:

- A hypothesis is `y = c * x1^p * x2^q`.
- A **mutation** nudges `p`, `q`, or both by ±0.25.
- `c` is fitted to the training rows.
- **Selection** keeps the four candidates with the lowest held-out error.

It starts from the naive law, `y ~ x1`, with no `x2`.

In [ ]:
!{sys.executable} "{ROOT}/lessons/scripts/02_openevolve/mini_evolve.py"

🔔 **Question:** The search stalls for a few generations before it finds a
better law. Why might changing one exponent at a time get stuck, when changing
both at once does not?

💡 **Tip:** Change `random.seed(5)` in `mini_evolve.py` to another number and
run it again to see a different path.

<a id='demo'></a>

# 🎬 Demo: OpenEvolve

With an API key, OpenEvolve lets a language model rewrite `fit()` itself. It
can invent any formula, not only power laws. The cell below runs 30 rounds and
skips itself when no key is set.

In [ ]:
openevolve_run = Path(sys.executable).parent / 'openevolve-run'
programs = f"{SCRIPTS / 'initial_program.py'} {SCRIPTS / 'evaluator.py'}"

if os.environ.get('OPENAI_API_KEY'):
    !{openevolve_run} {programs} --config {SCRIPTS / 'config.openai.yaml'} --iterations 30
elif os.environ.get('GEMINI_API_KEY'):
    !{openevolve_run} {programs} --config {SCRIPTS / 'config.yaml'} --iterations 30
else:
    print('No API key set: skipping. mini_evolve.py above is the offline version.')

In our test run with `gpt-4o-mini`, the score rose from 0.803 to 0.964. The
model found the same answer as Challenge 2: a power law fitted to the data.
A rate limit stopped it after 10 of 30 rounds.

<div class="alert alert-success">

## ❗ Key Points

* Evolution only improves what the evaluator measures, so the evaluator is the science.
* Score candidates on held-out groups, or memorizing the training data wins.
* A candidate program learns its constants from the training rows, then predicts unseen ones.
* Changing several parts of a hypothesis at once can escape a dead end.
* A language model makes the search more creative, but the same evaluator still decides.

</div>